In [ ]:
from langgraph.graph import StateGraph, END, START
from langchain_openai import ChatOpenAI
from typing import TypedDict, Literal, Annotated
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage

In [ ]:
load_dotenv()

In [ ]:
generator_llm = ChatOpenAI(model = "GPT-4.5")
evaluator_llm = ChatOpenAI(model = "GPT-4o-mini")
optimizer_llm = ChatOpenAI(model = "GPT-3o-mini")

In [ ]:
class tweetstate(TypedDict):
    
    topic : str
    tweet : str
    evaluation : Literal['approved','need_improvements']
    feedback : str
    iteration : str
    max_iteration : str

In [ ]:
from pydantic import BaseModel, Field

class tweetevaluation(BaseModel):
    evaluation : Literal['approved', 'need_improvement'] = Field(...,description = 'Final evalutaion')
    feedback : str = Field(...,description='feedback for the tweet')

In [ ]:
structured_evaluator_llm = evaluator_llm.with_structured_output(tweetevaluation)

In [ ]:
def generator(state : tweetstate):
    
    #prompt
    message = [SystemMessage(content = "You are a funny and clever Twitter/X influencer\n")
              HumanMessage(content = f"""
                           
                           write a short , original and hilarious tweeter post on topic : {state['topic']}
                           
                           Rules:
                           1. max 250 words
                           2. Do NOT use the Question/Answer format anywhere
                           3. use sarcasm , Irony, humour and cultural references.
                           4. Think in meme logic, punchlines or relatable takes.
                           5. use simple daya to day english
                           6. this is iteration no {state['iteration']+1}""")
              ]
    
    #LLM Generation
    response = generator_llm.invoke(message).content
    
    return {'tweet' : response}

def evaluator(state : tweetevaluation):
    
    message = [
        SystemMessage(content = f'you are a ruthless, no-laugh-given twitter critic. You evaluate tweets on the basis of how funny and influencing it is. The tweet is: {state["tweet"]}'),
        HumanMessage(content = f"""Evaluate this tweet and provide:
1. Whether it needs approval or improvements
2. Detailed feedback for improvement

Be critical and honest.""")
    ]
    
    response = structured_evaluator_llm.invoke(message)
    
    return {
        'evaluation': response.evaluation,
        'feedback': response.feedback
    }

In [ ]:
graph = StateGraph(tweetstate)

#create nodes
graph.add_node('generator', generator)
graph.add_node('evaluator', evaluator)
graph.add_node('optimizer',optimizer)
graph.add_node('')
graph.add_node('')
graph.add_node('')